<a href="https://colab.research.google.com/github/ElionLAB/OOP_2026_Practice/blob/main/ch_04/src/part_2/Lec_4_part_2_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Environment Setup — Auto-detect Google Colab / Local
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    pass  # This notebook does not require additional packages.
else:
    print('Local environment: Make sure `conda activate oop_practice` is active.')

# Lecture 4 — Part 2: Exceptions and Error Handling

**Konkuk University OOP (Python Object-Oriented Programming) — Spring 2026**

---

## Learning Objectives

1. Use `raise` to trigger built-in and custom exceptions
2. Understand call stack unwinding and unreachable code
3. Write `try / except / else / finally` blocks correctly
4. Navigate Python's exception hierarchy
5. Define custom exception classes with extra attributes
6. Compare EAFP vs LBYL and apply the Pythonic pattern
7. Build a data validation pipeline using exceptions (Iris case study)

## How to use this notebook

Each section has a **Full code reference** (shown first as a goal) and then a **TODO** cell where you fill in the blanks marked as `# TODO`. Run the test cell after each exercise to check your solution.

## 1. Triggering Exceptions & `raise` (Slides 4–6)

Python automatically raises exceptions for invalid operations (e.g., dividing by zero).
You can also **manually** raise exceptions with the `raise` keyword to enforce rules.

Key ideas:
- `raise` creates an exception instance and throws it.
- Normal execution **stops immediately** at the raise statement.
- Control transfers to the nearest exception handler on the call stack.

### Goal

1. Write `calculate_ratio` — a function that naturally raises `ZeroDivisionError`.
2. Write `check_type` — a function that manually raises `TypeError`.
3. Write `EvenOnly(list)` — a list subclass that raises on invalid `append` calls.

### Full code reference

```python
def calculate_ratio(x: int, y: int) -> float:
    return x / y  # Raises ZeroDivisionError if y == 0


def check_type(value: any) -> None:
    if not isinstance(value, int):
        raise TypeError("Only integers are allowed!")


class EvenOnly(list):
    def append(self, value: int) -> None:
        if not isinstance(value, int):
            raise TypeError("Only integers can be added")
        if value % 2 != 0:
            raise ValueError("Only even numbers can be added")
        super().append(value)
```

In [ ]:
# TODO 1: Complete the functions and the EvenOnly class.

def calculate_ratio(x: int, y: int) -> float:
    # TODO 1-a: return x divided by y (Python raises ZeroDivisionError automatically)
    ...


def check_type(value) -> None:
    # TODO 1-b: raise TypeError if value is not an int
    ...


class EvenOnly(list):
    def append(self, value: int) -> None:
        # TODO 1-c: raise TypeError if value is not an int
        ...
        # TODO 1-d: raise ValueError if value is odd
        ...
        # TODO 1-e: call the parent's append to actually add the value
        ...

In [ ]:
# Quick check
assert calculate_ratio(10, 2) == 5.0

try:
    calculate_ratio(10, 0)
except ZeroDivisionError:
    print("OK: ZeroDivisionError caught")
else:
    raise AssertionError("Should have raised ZeroDivisionError")

check_type(42)  # should not raise
try:
    check_type("hello")
except TypeError as e:
    print(f"OK: TypeError caught — {e}")
else:
    raise AssertionError("Should have raised TypeError")

eo = EvenOnly()
eo.append(2)
eo.append(4)
assert eo == [2, 4]

try:
    eo.append(3)
except ValueError:
    print("OK: ValueError for odd number")

try:
    eo.append("two")
except TypeError:
    print("OK: TypeError for non-int")

print("Section 1 passed.")

## 2. Call Stack Unwinding (Slides 7–8)

When an exception is raised:

1. **Trigger**: `raise` is executed.
2. **Halt**: The current function stops executing further lines.
3. **Bubble Up**: Control passes back to the calling function.
4. **Crash**: If no handler catches it, Python prints a Traceback and exits.

Any code **after** a `raise` in the same block is **unreachable** — it will never execute.

### Goal

Write `never_returns()` which always raises, and `call_exceptor()` which calls it.
Observe that the line after `never_returns()` is never reached.

### Full code reference

```python
from typing import NoReturn

def never_returns() -> NoReturn:
    print("I am about to raise an exception")
    raise Exception("This is always raised")
    print("This line will never execute")  # unreachable!


def call_exceptor() -> None:
    print("Call starts...")
    never_returns()
    print("Call ends...")  # unreachable!
```

In [ ]:
# TODO 2: Implement the two functions.
from typing import NoReturn


def never_returns() -> NoReturn:
    print("I am about to raise an exception")
    # TODO 2-a: raise an Exception with the message "This is always raised"
    ...
    print("This line will never execute")


def call_exceptor() -> None:
    print("Call starts...")
    # TODO 2-b: call never_returns()
    ...
    print("Call ends...")  # Unreachable!

In [ ]:
# Quick check
try:
    call_exceptor()
except Exception as e:
    print(f"OK, caught: {e}")
    assert str(e) == "This is always raised"
else:
    raise AssertionError("call_exceptor should have raised")

print("Section 2 passed.")

## 3. The `try...except` Syntax (Slides 10–11)

The `try...except` construct lets your program **survive errors gracefully**.

### Complete syntax

```python
try:
    # risky code
except SomeError:
    # handle that specific error
except AnotherError as e:
    # handle with access to the exception object
else:
    # runs only if NO exception occurred
finally:
    # ALWAYS runs (cleanup: close files, DB connections, etc.)
```

### Key rules
- `else` runs **only if no exception** was raised in `try`.
- `finally` **always runs**, even after a `return`.
- Use `finally` for **resource cleanup** (files, DB).

### Goal

Implement `funny_division` that handles `ZeroDivisionError`, `TypeError`, and a custom `ValueError` for 13.

### Full code reference

```python
def funny_division(divisor: int) -> float:
    try:
        if divisor == 13:
            raise ValueError("13 is unlucky")
        return 100 / divisor
    except ZeroDivisionError:
        return "Zero is not a good idea!"
    except TypeError as e:
        print(f"Caught a type error: {e}")
        return "Enter a numerical value"
```

In [ ]:
# TODO 3: Implement funny_division.

def funny_division(divisor: int) -> float:
    try:
        # TODO 3-a: if divisor is 13, raise ValueError("13 is unlucky")
        ...
        # TODO 3-b: return 100 / divisor
        ...
    # TODO 3-c: catch ZeroDivisionError and return "Zero is not a good idea!"
    except ZeroDivisionError:
        ...
    # TODO 3-d: catch TypeError as e, print the message, return "Enter a numerical value"
    except TypeError as e:
        ...

In [ ]:
# Quick check
assert funny_division(5) == 20.0
assert funny_division(0) == "Zero is not a good idea!"
assert funny_division("hello") == "Enter a numerical value"

try:
    funny_division(13)
except ValueError as e:
    print(f"OK: ValueError for 13 — {e}")
else:
    raise AssertionError("funny_division(13) should raise ValueError")

print("Section 3a passed.")

### The Complete `try` Block — `else` and `finally`

Now practice the full four-clause form.

### Full code reference

```python
def safe_divide(x, y):
    try:
        result = x / y
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None
    except TypeError as e:
        print(f"Wrong type: {e}")
        return None
    else:
        print(f"Success: {result}")
        return result
    finally:
        print("Cleanup done")
```

In [ ]:
# TODO 4: Write safe_divide with try / except / else / finally.

def safe_divide(x, y):
    try:
        # TODO 4-a: compute result = x / y
        ...
    except ZeroDivisionError:
        # TODO 4-b: print message and return None
        ...
    except TypeError as e:
        # TODO 4-c: print message and return None
        ...
    else:
        # TODO 4-d: print success message and return result
        ...
    finally:
        # TODO 4-e: print "Cleanup done"
        ...

In [ ]:
# Quick check
print("--- Test 1: normal ---")
assert safe_divide(10, 2) == 5.0

print("--- Test 2: zero ---")
assert safe_divide(10, 0) is None

print("--- Test 3: type error ---")
assert safe_divide(10, "a") is None

print("Section 3b passed.")

## 4. Custom Exception Classes (Slides 14–15)

If built-in exceptions lack the **semantic meaning** your application needs, create a **custom exception class**.

How to:
- Inherit from `Exception` (or a more specific subclass like `ValueError`).
- Override `__init__` to store extra context (amounts, balances, etc.).
- Add properties for computed attributes.

Benefits:
- Callers can catch your **specific exception type** and access its attributes.

### Goal

Build `InvalidWithdrawal(ValueError)` that stores `balance` and `amount`, and has an `overshoot` property.

### Full code reference

```python
class InvalidWithdrawal(ValueError):
    """Raised when withdrawal exceeds balance."""
    def __init__(self, balance: float, amount: float):
        super().__init__(f"Has {balance}, tried {amount}")
        self.balance = balance
        self.amount = amount

    @property
    def overshoot(self):
        return self.amount - self.balance
```

In [ ]:
# TODO 5: Implement InvalidWithdrawal.

class InvalidWithdrawal(ValueError):
    """Raised when withdrawal exceeds balance."""

    def __init__(self, balance: float, amount: float):
        # TODO 5-a: call super().__init__ with a descriptive message
        ...
        # TODO 5-b: store balance and amount as instance attributes
        self.balance = ...
        self.amount = ...

    @property
    def overshoot(self):
        # TODO 5-c: return how much the withdrawal exceeds the balance
        ...

In [ ]:
# Quick check
try:
    raise InvalidWithdrawal(balance=50.0, amount=80.0)
except InvalidWithdrawal as e:
    assert e.balance == 50.0
    assert e.amount == 80.0
    assert e.overshoot == 30.0
    assert isinstance(e, ValueError)  # it IS-A ValueError
    print(f"OK: {e}")
    print(f"Overshoot: {e.overshoot}")

print("Section 4 passed.")

## 5. EAFP vs LBYL (Slides 16–17)

Two coding philosophies for handling potential errors:

**LBYL — Look Before You Leap**
- Check conditions *before* execution (if/else guards).
- Verbose; checks and actions are separate (two accesses).

**EAFP — Easier to Ask Forgiveness than Permission**
- Just try the operation; catch exceptions if it fails.
- Clean, **Pythonic**, one access instead of two.
- Avoids TOCTOU race conditions in file I/O or threading.

**Verdict**: Python community strongly prefers **EAFP**.

### Goal

Implement inventory management using the EAFP pattern with custom exceptions.

### Full code reference

```python
class InvalidItemType(Exception):
    pass

class OutOfStock(Exception):
    pass


def sell_item_lbyl(inventory: dict, item_name: str) -> None:
    """LBYL: Look Before You Leap (messy)"""
    if item_name in inventory:
        if inventory[item_name] > 0:
            inventory[item_name] -= 1
        else:
            raise OutOfStock(item_name)
    else:
        raise InvalidItemType(item_name)


def sell_item_eafp(inventory: dict, item_name: str) -> None:
    """EAFP: Easier to Ask Forgiveness than Permission (Pythonic)"""
    try:
        inventory[item_name] -= 1
    except KeyError:
        raise InvalidItemType(item_name)
    if inventory[item_name] < 0:
        inventory[item_name] += 1  # undo
        raise OutOfStock(item_name)
```

In [ ]:
# TODO 6: Implement EAFP-style inventory management.

class InvalidItemType(Exception):
    pass


class OutOfStock(Exception):
    pass


def sell_item_eafp(inventory: dict, item_name: str) -> None:
    """EAFP: Easier to Ask Forgiveness than Permission (Pythonic)"""
    try:
        # TODO 6-a: try to decrement inventory[item_name]
        ...
    except KeyError:
        # TODO 6-b: raise InvalidItemType with item_name
        ...
    # TODO 6-c: if count went below 0, undo and raise OutOfStock
    ...

In [ ]:
# Quick check
inv = {"apple": 2, "banana": 0}

sell_item_eafp(inv, "apple")
assert inv["apple"] == 1
print("OK: sold an apple")

try:
    sell_item_eafp(inv, "banana")
except OutOfStock as e:
    print(f"OK: OutOfStock — {e}")
    assert inv["banana"] == 0  # undo worked
else:
    raise AssertionError("Should have raised OutOfStock")

try:
    sell_item_eafp(inv, "cherry")
except InvalidItemType as e:
    print(f"OK: InvalidItemType — {e}")
else:
    raise AssertionError("Should have raised InvalidItemType")

print("Section 5 passed.")

## 6. Case Study — Iris Data Validation Pipeline (Slides 18–28)

A **k-Nearest-Neighbors** classifier needs valid training data. Raw CSV rows may contain:
- **Missing data** or **non-numeric strings** (e.g., `"N/A"`) for measurements.
- **Invalid species names** that don't match the three known Iris types.

### Strategy
Use Python exceptions to **discover** exceptional data (`raise`) and **respond** to it (`catch` and report).

### Architecture
- `InvalidSampleError(ValueError)` — custom exception for bad CSV rows.
- `Sample` — stores 4 float measurements; `from_dict` class method validates and converts.
- `KnownSample(Sample)` — adds a validated `species` field.
- `TrainingData` — loads CSV rows, catches `InvalidSampleError` per row, reports and skips bad data.

### Key patterns used
- `raise ... from` to chain exceptions (preserve the original cause).
- `@classmethod` factory so subclasses inherit the parser for free.
- Set-based categorical validation for species.

### Full code reference

```python
class InvalidSampleError(ValueError):
    pass


class Sample:
    def __init__(
        self,
        sepal_length: float,
        sepal_width: float,
        petal_length: float,
        petal_width: float,
    ) -> None:
        self.sepal_length = sepal_length
        self.sepal_width = sepal_width
        self.petal_length = petal_length
        self.petal_width = petal_width

    @classmethod
    def from_dict(cls, row: dict) -> 'Sample':
        try:
            return cls(
                sepal_length=float(row["sepal_length"]),
                sepal_width=float(row["sepal_width"]),
                petal_length=float(row["petal_length"]),
                petal_width=float(row["petal_width"]),
            )
        except ValueError as ex:
            raise InvalidSampleError(f"Invalid: {row!r}") from ex


VALID_SPECIES = {"Iris-setosa", "Iris-versicolor", "Iris-virginica"}


class KnownSample(Sample):
    def __init__(self, species: str = "", **kwargs) -> None:
        super().__init__(**kwargs)
        self.species = species

    @classmethod
    def from_dict(cls, row: dict) -> 'KnownSample':
        if row["species"] not in VALID_SPECIES:
            raise InvalidSampleError(
                f"Invalid species: {row['species']}")
        try:
            return cls(
                species=row["species"],
                sepal_length=float(row["sepal_length"]),
                sepal_width=float(row["sepal_width"]),
                petal_length=float(row["petal_length"]),
                petal_width=float(row["petal_width"]),
            )
        except ValueError as ex:
            raise InvalidSampleError(f"Invalid: {row!r}") from ex


class TrainingData:
    def __init__(self) -> None:
        self.samples: list[KnownSample] = []

    def load(self, raw_data_iter) -> None:
        for n, row in enumerate(raw_data_iter):
            try:
                sample = KnownSample.from_dict(row)
                self.samples.append(sample)
            except InvalidSampleError as ex:
                print(f"Row {n+1} skipped: {ex}")
```

In [ ]:
# TODO 7: Build the Iris data validation pipeline.

# TODO 7-a: Define InvalidSampleError as a subclass of ValueError
class InvalidSampleError(ValueError):
    ...


class Sample:
    def __init__(
        self,
        sepal_length: float,
        sepal_width: float,
        petal_length: float,
        petal_width: float,
    ) -> None:
        self.sepal_length = sepal_length
        self.sepal_width = sepal_width
        self.petal_length = petal_length
        self.petal_width = petal_width

    @classmethod
    def from_dict(cls, row: dict) -> 'Sample':
        try:
            # TODO 7-b: return cls(...) with float() conversions for all 4 fields
            ...
        except ValueError as ex:
            # TODO 7-c: raise InvalidSampleError with "raise ... from ex"
            ...


# TODO 7-d: define the set of valid species
VALID_SPECIES = ...


class KnownSample(Sample):
    def __init__(self, species: str = "", **kwargs) -> None:
        # TODO 7-e: call super().__init__(**kwargs) and store species
        ...
        self.species = species

    @classmethod
    def from_dict(cls, row: dict) -> 'KnownSample':
        # TODO 7-f: validate species against VALID_SPECIES; raise InvalidSampleError if invalid
        ...
        try:
            # TODO 7-g: return cls(...) with species + float conversions
            ...
        except ValueError as ex:
            raise InvalidSampleError(f"Invalid: {row!r}") from ex


class TrainingData:
    def __init__(self) -> None:
        self.samples: list[KnownSample] = []

    def load(self, raw_data_iter) -> None:
        for n, row in enumerate(raw_data_iter):
            try:
                # TODO 7-h: create a KnownSample from the row and append it
                ...
            except InvalidSampleError as ex:
                # TODO 7-i: print which row was skipped and why
                ...

In [ ]:
# Quick check
test_data = [
    {"sepal_length": "5.1", "sepal_width": "3.5",
     "petal_length": "1.4", "petal_width": "0.2", "species": "Iris-setosa"},
    {"sepal_length": "N/A", "sepal_width": "3.0",
     "petal_length": "4.5", "petal_width": "1.5", "species": "Iris-versicolor"},
    {"sepal_length": "6.3", "sepal_width": "3.3",
     "petal_length": "6.0", "petal_width": "2.5", "species": "Iris-virginica"},
    {"sepal_length": "5.0", "sepal_width": "3.4",
     "petal_length": "1.5", "petal_width": "0.2", "species": "Iris-unknown"},
]

td = TrainingData()
td.load(test_data)

# Row 0: valid (Iris-setosa)
# Row 1: invalid float ("N/A") — skipped
# Row 2: valid (Iris-virginica)
# Row 3: invalid species ("Iris-unknown") — skipped
assert len(td.samples) == 2, f"Expected 2 valid samples, got {len(td.samples)}"
assert td.samples[0].species == "Iris-setosa"
assert td.samples[0].sepal_length == 5.1
assert td.samples[1].species == "Iris-virginica"
print(f"Loaded {len(td.samples)} valid samples (2 skipped).")
print("Section 6 passed.")

## 7. Exercises

Three short exercises to consolidate everything above. Each has a `# TODO` to fill and a test cell.

### Exercise 1 — Custom exception with extra attributes

Build `InsufficientFundsError(Exception)` for a banking application.

- `__init__(self, balance, amount)` stores both values and calls `super().__init__` with a message.
- `deficit` property returns `amount - balance`.

Then write a `withdraw(balance, amount)` function that raises `InsufficientFundsError` when `amount > balance`, or returns the new balance.

In [ ]:
# TODO Exercise 1

class InsufficientFundsError(Exception):
    def __init__(self, balance: float, amount: float):
        # TODO E1-a: call super().__init__ with a descriptive message
        ...
        # TODO E1-b: store balance and amount
        ...

    @property
    def deficit(self) -> float:
        # TODO E1-c: return amount - balance
        ...


def withdraw(balance: float, amount: float) -> float:
    # TODO E1-d: raise InsufficientFundsError if amount > balance
    ...
    # TODO E1-e: return new balance
    ...

In [ ]:
# Quick check
assert withdraw(100, 30) == 70

try:
    withdraw(50, 80)
except InsufficientFundsError as e:
    assert e.balance == 50
    assert e.amount == 80
    assert e.deficit == 30
    print(f"OK: {e}")
    print(f"Deficit: {e.deficit}")
else:
    raise AssertionError("Should have raised InsufficientFundsError")

print("Exercise 1 OK.")

### Exercise 2 — EAFP file reader

Write two versions of a function that reads a file and returns its contents:

1. **LBYL** version: check `os.path.exists` first.
2. **EAFP** version: just `open()`, catch `FileNotFoundError`.

Both should return `None` if the file doesn't exist.

> **Hints** — this exercise uses a few Python features **not covered in the lecture**:
>
> - **`with open(path) as f:`** — a *context manager*. It automatically closes the file when the block exits (even on exception), so you don't need a `finally: f.close()`. Read the whole file with `f.read()`. Mini-example:
>   ```python
>   with open("greeting.txt") as f:
>       text = f.read()
>   # file is already closed here
>   ```
> - **`os.path.exists(path)`** — returns `True` if the path exists, `False` otherwise. Use it for the LBYL check.
> - **`str | None`** in the return type — PEP 604 union syntax meaning "returns a `str` **or** `None`". Requires Python 3.10+. On Python 3.9, either add `from __future__ import annotations` at the top of the cell, or use `Optional[str]` from `typing` instead.

In [ ]:
# TODO Exercise 2
import os


def read_file_lbyl(path: str) -> str | None:
    # TODO E2-a: check os.path.exists(path), then open and read
    ...


def read_file_eafp(path: str) -> str | None:
    # TODO E2-b: try to open and read, catch FileNotFoundError
    ...

In [ ]:
# Quick check — create a temporary file for testing
import tempfile

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write("hello OOP")
    tmp_path = f.name

# Existing file
assert read_file_lbyl(tmp_path) == "hello OOP"
assert read_file_eafp(tmp_path) == "hello OOP"

# Non-existing file
assert read_file_lbyl("/tmp/no_such_file_12345.txt") is None
assert read_file_eafp("/tmp/no_such_file_12345.txt") is None

os.unlink(tmp_path)
print("Exercise 2 OK.")

### Exercise 3 — Guard method (precondition check)

Extend `TrainingData` from Section 6 with a `classify` method that:
- Raises `RuntimeError` if `load()` hasn't been called yet (i.e., `self.samples` is empty).
- Otherwise, returns the species of the **first** sample (a placeholder for real k-NN logic).

This demonstrates the **guard pattern** — checking preconditions and raising if violated (Slide 22).

In [ ]:
# TODO Exercise 3

class TrainingDataV2(TrainingData):
    def classify(self, unknown_sample: dict) -> str:
        # TODO E3-a: raise RuntimeError if no samples are loaded
        ...
        # TODO E3-b: return self.samples[0].species (placeholder for real k-NN)
        ...

In [ ]:
# Quick check
td2 = TrainingDataV2()

# Before loading — should raise
try:
    td2.classify({"sepal_length": "5.0"})
except RuntimeError as e:
    print(f"OK: guard caught — {e}")
else:
    raise AssertionError("Should have raised RuntimeError")

# After loading
td2.load([
    {"sepal_length": "5.1", "sepal_width": "3.5",
     "petal_length": "1.4", "petal_width": "0.2", "species": "Iris-setosa"},
])
result = td2.classify({"sepal_length": "5.0"})
assert result == "Iris-setosa"
print(f"OK: classified as {result}")
print("Exercise 3 OK.")

---

## Summary

- **`raise`** stops normal execution and transfers control to the nearest exception handler.
- **`try / except / else / finally`** is the full exception-handling construct: `else` runs on success, `finally` always runs.
- Python's **exception hierarchy** lets you catch broad or specific errors: always catch `Exception`, never `BaseException`.
- **Custom exceptions** inherit from `Exception` and can carry domain-specific attributes (balance, amount, etc.).
- **EAFP** (try first, catch on failure) is more Pythonic than **LBYL** (check first, act second).
- In data validation, use `raise ... from` to **chain exceptions** and keep the original cause visible.

*"Exceptions are a Powerful Communication Tool."*